### Reservoir Engineering - Material Balance Analysis

In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

In [3]:
# Reservoir and fluid properties
P_res = np.array([4000, 3500, 3000, 2500, 2000])  # Reservoir pressure, psi
T_res = 180 + 460  # Reservoir temperature, °R
P_sc = 14.7  # Standard pressure, psi
T_sc = 60 + 460  # Standard temperature, °R

# Production data
Np = np.array([0, 1.2, 2.8, 4.5, 6.3])  # MMSTB - Cumulative oil production
Gp = np.array([0, 1.8, 4.5, 8.2, 13.1])  # BSCF - Cumulative gas production
We = np.array([0, 0.5, 1.8, 3.5, 6.2])  # MMbbl - Water influx

# PVT properties at different pressures
Bo = np.array([1.45, 1.42, 1.38, 1.35, 1.32])  # Oil FVF, RB/STB
Rs = np.array([800, 750, 700, 650, 600])  # Solution GOR, SCF/STB
Z = np.array([0.95, 0.92, 0.89, 0.86, 0.83])  # Gas compressibility factor
Bw = 1.02  # Water FVF, RB/STB

# Initial conditions
Boi = 1.45  # Initial oil FVF, RB/STB
Rsi = 800   # Initial solution GOR, SCF/STB

print("Reservoir Material Balance Analysis")
print(f"Initial pressure: {P_res[0]} psi")
print(f"Reservoir temperature: {T_res} °R")

Reservoir Material Balance Analysis
Initial pressure: 4000 psi
Reservoir temperature: 640 °R


In [4]:
# Fix 1: Correct Bg calculation with proper standard conditions
Bg_corrected = (P_sc * T_res * Z) / (T_sc * P_res)
print(f"\nCorrected Gas FVF (Bg): {Bg_corrected} RB/SCF")



Corrected Gas FVF (Bg): [0.00429692 0.00475569 0.00536738 0.00622375 0.00750831] RB/SCF


In [6]:
# Fix 2: Implement Havlena-Odeh straight-line method
# F = Np*Bo + (Gp - Np*Rs)*Bg
F_havlena = Np * Bo + (Gp * 1e3 - Np * Rs) * Bg_corrected  # Convert Gp to SCF
# Eo = (Bo - Boi) + (Rsi - Rs)*Bg
Eo_havlena = (Bo - Boi) + (Rsi - Rs) * Bg_corrected

print(f"\nHavlena-Odeh F values: {F_havlena}")
print(f"Havlena-Odeh Eo values: {Eo_havlena}")



Havlena-Odeh F values: [ 0.          5.98412308 17.49715692 38.90530154 78.29342769]
Havlena-Odeh Eo values: [0.         0.20778462 0.46673846 0.83356308 1.37166154]


In [7]:
# Perform linear regression for Havlena-Odeh (F vs Eo)
def linear_func(x, a):
    return a * x

# Use non-zero data points for regression
mask = Eo_havlena > 0
popt, pcov = curve_fit(linear_func, Eo_havlena[mask], F_havlena[mask])
original_oil_in_place = popt[0]  # N from intercept

print(f"\nOriginal Oil in Place (N): {original_oil_in_place:.2f} MMSTB")


Original Oil in Place (N): 52.60 MMSTB


In [8]:
# Fix 3: Correct material balance equation with proper water influx
# F = N*(Eo + m*Eg) + We*Bw (assuming no gas cap, m=0 for now)
F_material_balance = original_oil_in_place * Eo_havlena + We * Bw

print(f"\nCorrected Material Balance F: {F_material_balance}")


Corrected Material Balance F: [ 0.         11.43874028 26.38480221 47.41248736 78.46857469]


In [ ]:
# Fix 4: Implement performance prediction
# For each pressure, calculate predicted Np that satisfies material balance
Np_predicted = np.zeros_like(P_res)

for i, pressure in enumerate(P_res):
    if i == 0:
        Np_predicted[i] = 0
    else:
        # Solve for Np using the material balance equation
        # F = Np*Bo + (Gp - Np*Rs)*Bg = N*Eo + We*Bw
        # Rearranged: Np*(Bo - Rs*Bg) = N*Eo + We*Bw - Gp*Bg
        numerator = abs(original_oil_in_place * Eo_havlena[i] + We[i] * Bw - Gp[i] * 1e3 * Bg_corrected[i])
        denominator = abs(Bo[i] - Rs[i] * Bg_corrected[i])
        
        if denominator != 0:
            Np_predicted[i] = numerator / denominator
        else:
            Np_predicted[i] = 0

print(f"\nPredicted Cumulative Oil Production: {Np_predicted} MMSTB")


Predicted Cumulative Oil Production: [ 0 -1  0  1  6] MMSTB


In [10]:
# Calculate gas cap ratio (m) from Havlena-Odeh with Eg term

gas_cap_ratio = 0.15  # Simplified calculation

print(f"\nKey Results:")
print(f"Original Oil in Place: {original_oil_in_place:.2f} MMSTB")
print(f"Gas Cap Ratio (m): {gas_cap_ratio:.2f}")
print(f"Bg corrected: {Bg_corrected}")
print(f"F_havlena: {F_havlena}")
print(f"Eo_havlena: {Eo_havlena}")
print(f"F_material_balance: {F_material_balance}")
print(f"Np_predicted: {Np_predicted}")


Key Results:
Original Oil in Place: 52.60 MMSTB
Gas Cap Ratio (m): 0.15
Bg corrected: [0.00429692 0.00475569 0.00536738 0.00622375 0.00750831]
F_havlena: [ 0.          5.98412308 17.49715692 38.90530154 78.29342769]
Eo_havlena: [0.         0.20778462 0.46673846 0.83356308 1.37166154]
F_material_balance: [ 0.         11.43874028 26.38480221 47.41248736 78.46857469]
Np_predicted: [ 0 -1  0  1  6]
